# Phase 5: Model Setup

Goal of this phase: load the actual pretrained BiomedCLIP checkpoint and decide how to fine-tune it, full fine-tune versus freezing parts of the encoders. I flagged back in phase 4 that the RTX 3060 only has 6GB of VRAM, which is tight for CLIP style contrastive fine-tuning, so I want this decision backed by real measured numbers rather than a guess.

All of the actual model loading, freezing and benchmarking code lives in src/models/biomedclip.py (load_biomedclip, set_freeze_mode) and src/models/benchmark.py (measure_memory, time_iterations), not in this notebook. This notebook records what I ran, what came back, and why I decided what I decided.

## The model itself

Loaded via src/models/biomedclip.py::load_biomedclip, which wraps open_clip's create_model_and_transforms for the BiomedCLIP-PubMedBERT_256-vit_base_patch16_224 checkpoint. It returns an open_clip CustomTextCLIP with two top level pieces, visual and text, plus a learned logit_scale temperature parameter.

Parameter counts:

- visual.trunk (ViT-B/16 backbone, 12 blocks): 85,798,656
- visual.head (projection): 393,216
- text.transformer (PubMedBERT, 12 layers): 108,891,648
- text.proj (projection): 819,200
- total: 195,902,721

The pretrained logit_scale starts at 4.4454, close to the ln(100) = 4.6052 clamp value used in the original CLIP paper, meaning the pretrained model already learned a fairly sharp temperature.

## Measuring real GPU memory for full, frozen and partial fine-tuning

Rather than guess whether full fine-tuning fits on 6GB, I ran src/models/benchmark.py directly as a script (python -m src.models.benchmark), which does an actual forward pass, backward pass and optimizer step for each freeze mode at batch size 32, and reads off the real peak memory.

GPU: NVIDIA GeForce RTX 3060 Laptop GPU, total VRAM: 6.44 GB

| mode | batch | trainable params | peak VRAM | status |
|---|---|---|---|---|
| full | 32 | 195,902,721 (100.0%) | 9.87 GB | ok, but only via slow spillover, see below |
| frozen_backbone | 32 | 1,212,417 (0.6%) | 1.12 GB | ok |
| partial | 32 | 29,563,905 (15.1%) | 2.49 GB | ok |

Full fine-tuning needs 9.87GB, which is more than the entire 6.44GB the GPU has. Not a close call, full fine-tuning is simply not possible on this hardware at any reasonable batch size without something like gradient checkpointing or heavy accumulation, and I would rather not go down that path for a first fine-tune. Frozen backbone (only the projection heads and logit_scale trainable) needs just 1.12GB. Partial (last 2 transformer blocks of each encoder plus both heads, 15.1 percent of params) needs 2.49GB. Both of the latter two comfortably fit, with a lot of headroom left over, which is worth using since bigger batches generally help contrastive learning through more in-batch negatives.

One thing worth noting about that 9.87GB number for full fine-tuning: on this Windows setup, going over budget does not actually raise an out of memory error, it silently spills into shared system RAM through the driver and just runs very slowly instead of failing fast. So "ok" status there does not mean it is actually usable, just that it did not crash.

## Checking how much batch size headroom is actually usable

Swept a few larger batch sizes for the two viable modes with the same script.

| mode | batch | peak VRAM | status |
|---|---|---|---|
| partial | 64 | 4.13 GB | ok |
| partial | 96 | 5.79 GB | ok |
| partial | 128 | 7.44 GB | ok, but same spillover concern as full above |
| frozen_backbone | 128 | 2.08 GB | ok |
| frozen_backbone | 256 | 3.35 GB | ok |
| frozen_backbone | 384 | 4.62 GB | ok |

Partial at batch 128 reporting 7.44GB "ok" on a 6.44GB GPU is the same spillover situation as full fine-tuning above, not a real success. Even batch 96, which does fit within the nominal 6.44GB at 5.79GB, was suspicious enough (that close to the ceiling) that I did not want to just trust the raw allocation number.

## Verifying the suspicion with real throughput

Timed actual iterations (forward, backward, optimizer step) for partial fine-tuning at batch 64 versus batch 96, using src/models/benchmark.py::time_iterations, to see if batch 96 pays a hidden performance cost despite technically fitting.

| mode | batch | ms per sample | samples per second |
|---|---|---|---|
| partial | 64 | 41.79 | 23.9 |
| partial | 96 | 76.72 | 13.0 |

Confirmed. Batch 96 very nearly doubles the per sample time compared to batch 64, despite only using 5.79GB out of a nominal 6.44GB. So even a batch size that technically fits according to the allocator is already paying a real throughput penalty, presumably because the OS and driver reserve some VRAM for other things so the truly free amount is less than the nominal total. Batch 64 for partial fine-tuning is the real safe ceiling in plain fp32, not just whatever number happens to not crash.

Side note on how I got this number: my first two attempts at timing this inside a live notebook cell in this same kernel session both timed out (900 seconds, then 1200 seconds), even though the identical code finished in under a minute as a standalone script. My read is that this notebook's kernel had already run several heavy model loading operations earlier in the session, and GPU memory fragmentation seems to build up across cells within one long lived process in a way that does not happen when each run starts fresh. That is part of why the actual benchmarking code belongs in src/models/benchmark.py and gets run as a standalone script rather than live in a notebook cell, it is also just more reliable that way, not only better organized. Worth keeping in mind for phase 6 too, since the real training script will also be one long lived process from start to finish.

## What existing work does here

I did not want to pick full versus frozen versus partial in a vacuum. Two points from prior work directly support going with a partial fine-tune rather than full or fully frozen.

Zhang et al. (2022), the ConVIRT paper I already used in phase 3 for the sentence sampling discussion, does exactly this kind of partial freeze on their text encoder: they keep the pretrained weights for the first 6 layers of a BERT encoder and only fine-tune the last 6 layers for their contrastive task, rather than fine-tuning the whole thing or freezing it entirely.

A CS231N course project, "Parameter-Efficient Fine-Tuning of BiomedCLIP for Diabetic Retinopathy" (Stanford CS231N, 2025), fine-tunes this exact BiomedCLIP checkpoint on a downstream medical task using LoRA and BitFit, both parameter-efficient approaches that only touch a small fraction of the weights, specifically because of similar computational constraints. Their setup: AdamW, learning rate 1e-5, 1 epoch, batch size 32. I am treating this as a directional reference point rather than a rule, since it is a student project report and not peer reviewed, but the practical conclusion, fine-tune a small subset of BiomedCLIP's weights rather than the whole thing, matches what my own memory measurements are telling me independently.

## Decision: partial fine-tuning, last 2 blocks of each encoder, batch size 64

Full fine-tuning is not an option on this GPU, that part was not really a judgment call. Between frozen backbone and partial, I am going with partial: freeze everything except the last 2 transformer blocks of the visual trunk, the last 2 layers of the text transformer, both projection heads, and logit_scale. That is 29.6 million trainable parameters out of 195.9 million, 15.1 percent.

My reasoning for not just going with the frozen backbone option, even though it is cheaper and would allow a much bigger batch size: frozen backbone only adapts the projection heads, meaning the underlying image and text representations stay exactly as BiomedCLIP learned them from its own pretraining corpus. My dataset is a real domain shift from that, a specific hospital's imaging equipment and reporting style, plus translated rather than natively English text, so I want the encoders themselves to have some ability to adapt, not just the final projection. Partial gives me that while still fitting comfortably in memory at a reasonable batch size, and it mirrors what ConVIRT itself does for its text encoder specifically.

## Implementation

load_biomedclip and set_freeze_mode live in src/models/biomedclip.py. Checked the freeze boundary lands exactly where intended: blocks 0-9 of each encoder frozen, blocks 10-11 trainable, both projection heads and logit_scale trainable, 29,563,905 trainable parameters out of 195,902,721 total (15.1 percent), matching the number measured above.

## Summary and what's next

Loaded the actual pretrained BiomedCLIP checkpoint and measured, rather than guessed, what fits on the RTX 3060's 6GB. Full fine-tuning needs 9.87GB and is not possible here. Frozen backbone and partial fine-tuning both fit, and after checking batch size headroom (and catching two cases, full fine-tuning and partial at batch 128, that technically did not crash but were secretly relying on slow shared memory spillover) I settled on partial fine-tuning, last 2 blocks of each encoder plus both projection heads, at batch size 64, backed by ConVIRT's own precedent of partially freezing its text encoder.

Implemented as load_biomedclip and set_freeze_mode in src/models/biomedclip.py, with the actual benchmarking harness in src/models/benchmark.py, runnable standalone via python -m src.models.benchmark.

Next phase: the actual training loop, contrastive loss, optimizer, learning rate schedule, checkpointing and logging. I did some upfront research into this already (loss function, logit_scale clamping convention, learning rate and warmup ranges used for fine-tuning versus pretraining, mixed precision as a possible way to push batch size further than what I measured here in plain fp32) which I'm keeping as a separate research note rather than folding into this notebook, since none of it is actually implemented or decided yet.

## Addendum: bf16 lets me use a much bigger batch than plain fp32

Everything above was measured in plain fp32. Before locking in batch size 64 as final, I tried bf16 automatic mixed precision (the RTX 3060 supports it natively, and it is what BiomedCLIP's own pretraining used) via src/models/benchmark.py's amp_dtype argument, standalone script again, not live in this notebook.

| mode | batch | precision | peak VRAM | per sample |
|---|---|---|---|---|
| partial | 64 | bf16 | 2.94 GB | 13.01ms |
| partial | 96 | bf16 | 4.00 GB | 12.97ms |
| partial | 128 | bf16 | 5.04 GB | 12.87ms |
| partial | 160 | bf16 | 6.07 GB | 32.46ms |
| full | 32 | bf16 | 6.71 GB | not tested further |

bf16 roughly triples throughput at batch 64 (13ms per sample versus 41.79ms in fp32) and, more importantly, batch 64, 96 and 128 all cost essentially the same per sample time under bf16, meaning the fp32 throughput cliff I hit at batch 96 simply is not there anymore until batch 160, where the same kind of memory pressure cliff reappears (32.46ms, back to fp32-96 territory). So bf16 does not just speed things up, it roughly doubles the real usable batch size ceiling, 128 instead of 64.

Full fine-tuning at batch 32 drops from 9.87GB in fp32 to 6.71GB with bf16, a real reduction, but still over the 6.44GB available, so it still does not fit. A much smaller batch might, but I did not chase that further since partial fine-tuning with bf16 at batch 128 is already a strong setup and a small full-finetune batch would hurt the contrastive loss's in-batch negative count more than it is worth.

Updating my actual phase 6 starting point because of this: partial fine-tuning, bf16 autocast, batch size 128, not the fp32 batch 64 I had concluded above before testing this.